# Chemprop v2 — Experiment 1: Default Parity Encoding

This notebook tests Chemprop's default `SimpleMoleculeMolGraphFeaturizer`, which
encodes the `@/@@` SMILES parity tag via RDKit's `ChiralTag` atom feature (values:
`CHI_UNSPECIFIED`, `CHI_TETRAHEDRAL_CW`, `CHI_TETRAHEDRAL_CCW`). This is
**different** from Morgan fingerprints with `includeChirality=True`, which encode
the **CIP R/S label** (confirmed by Greg Landrum, GitHub Issues #2388 and #4013).

**Scientific questions:** what stereochemical signal does the MPNN extract from
the 2D graph, and does it survive the permutation-invariant mean aggregation?

**Conditions in this notebook:**

| Condition | Type | Datapoints | Description |
|---|---|---|---|
| `exp_1_default_parity_singletask` | Single-task | `all_data_stereo` | Default featurizer, implicit Hs |
| `exp_1_default_parity_multitask` | Multi-task | `all_data_stereo` | Default featurizer, implicit Hs, joint 3-target model |

**Total fits:**
- Singletask: 1 condition × 3 targets × 5 folds = 15
- Multitask: 5 folds = 5
- **Total: 20 fits**

**Hypotheses tested:**
1. `@/@@` should be the easiest target for Chemprop (inverse of the RF result), because Chemprop encodes `@/@@` via ChiralTag while the RF encoded CIP R/S.
2. `@/@@` signal should mostly survive the aggregation.
3. Multitask training may slightly improve per-target performance through shared MPNN representations.

Part of series: `3_RF_morgan_count.ipynb` → `4_chemprop_exp_0.ipynb` → **`4_chemprop_exp_1.ipynb`**

In cheminformatics, the @ and @@ symbols in a SMILES string represent the parity of the stereocenter. Parity is a relative directional tag; it describes the spatial arrangement (@ for counter-clockwise and @@ for clockwise) strictly based on the order the atoms are written in the SMILES string.

## Imports

In [1]:
import gc
import time
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

import numpy as np
import pandas as pd
import torch

from lightning import pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint

from sklearn.metrics import (
    roc_auc_score,
    matthews_corrcoef,
    accuracy_score,
    f1_score,
    average_precision_score,
)

from chemprop import data, featurizers, models, nn

## Configuration

Chemprop's MoleculeDataset uses PyTorch's DataLoader object
- https://github.com/chemprop/chemprop/blob/7033cac6ebc1a66f885705c7fa3e2cb960f96e49/chemprop/data/datasets.py#L189

- `num_workers=0`: data loaded sequentially in the main process (efficient for this dataset size).

In [2]:
INPUT_PATH  = 'data/class_all.csv'
FOLDS_PATH  = 'data/cmrt_folds.npz'
NUM_WORKERS = 0      # set >0 if multiprocessing is available
MAX_EPOCHS  = 50     # upper bound; EarlyStopping will typically stop sooner
PATIENCE    = 10     # early stopping patience
BATCH_SIZE  = 64
SEED        = 42

pl.seed_everything(SEED, workers=True)

Seed set to 42


42

## Load data and splits

In [3]:
df = pd.read_csv(INPUT_PATH, index_col=0)
print(f'Dataset shape: {df.shape}')
df.head()

Dataset shape: (3858, 6)


,SMILES,SMILES_opp,TR/TE,F/L_class,@/@@_class,R/S_class
0,Brc1ccc2c(c1)N[C@H](c1ccccc1)CC2,Brc1ccc2c(c1)N[C@@H](c1ccccc1)CC2,TE,F,@,S
1,Brc1ccc2c(c1)N[C@@H](c1ccccc1)CC2,Brc1ccc2c(c1)N[C@H](c1ccccc1)CC2,TE,L,@@,R
2,C#CCO[C@H](CSc1nc2cc(Cl)ccc2s1)CN(C)C(c1ccccc1...,C#CCO[C@@H](CSc1nc2cc(Cl)ccc2s1)CN(C)C(c1ccccc...,TE,F,@,S
3,C#CCO[C@@H](CSc1nc2cc(Cl)ccc2s1)CN(C)C(c1ccccc...,C#CCO[C@H](CSc1nc2cc(Cl)ccc2s1)CN(C)C(c1ccccc1...,TE,L,@@,R
4,C=C(C(C)=O)[C@@H](CC(=O)c1ccc(Br)cc1)C(=O)OCC,C=C(C(C)=O)[C@H](CC(=O)c1ccc(Br)cc1)C(=O)OCC,TE,F,@@,R


In [4]:
def load_folds(path: str) -> list[dict[str, np.ndarray]]:
    """
    Load pre-computed splits from a .npz file.

    Args:
        path: Path to .npz file saved by save_folds() in notebook 2.

    Returns:
        List of dicts with keys 'train', 'val', 'test' as numpy arrays
        of molecule-level integer indices into the full dataset.
    """
    archive = np.load(path)
    fold_indices = sorted(set(
        int(k.split('_')[0].replace('fold', '')) for k in archive.files))
    return [
        {
            'train': archive[f'fold{i}_train'],
            'val':   archive[f'fold{i}_val'],
            'test':  archive[f'fold{i}_test'],
        }
        for i in fold_indices
    ]

In [5]:
mol_folds = load_folds(FOLDS_PATH)
print(f'Loaded {len(mol_folds)} folds')
for i, fold in enumerate(mol_folds):
    n_tr = len(fold['train'])
    n_v  = len(fold['val'])
    n_te = len(fold['test'])
    print(f'  Fold {i}: train={n_tr:,}  val={n_v:,}  test={n_te:,}')

Loaded 5 folds
  Fold 0: train=3,478  val=190  test=190
  Fold 1: train=3,478  val=190  test=190
  Fold 2: train=3,478  val=190  test=190
  Fold 3: train=3,478  val=190  test=190
  Fold 4: train=3,478  val=190  test=190


## Target variables and label encoding

All three targets are binary and perfectly balanced (50/50). Chance baseline: 50%.

In [6]:
TARGET_LABEL_MAPS = {
    'R/S_class':  {'R': 0, 'S': 1},
    '@/@@_class': {'@': 0, '@@': 1},
    'F/L_class':  {'F': 0, 'L': 1},
}
SINGLE_TARGETS = list(TARGET_LABEL_MAPS.keys())

for col, lmap in TARGET_LABEL_MAPS.items():
    df[f'label_{col}'] = df[col].map(lmap)

print('Class distributions (all should be 50/50):')
for col in SINGLE_TARGETS:
    vc = df[f'label_{col}'].value_counts(normalize=True)
    print(f'  {col}: {vc.to_dict()}')

Class distributions (all should be 50/50):
  R/S_class: {1: 0.5, 0: 0.5}
  @/@@_class: {0: 0.5, 1: 0.5}
  F/L_class: {0: 0.5, 1: 0.5}


## Featurizer: default parity encoding

### Exp 1 — Default Chemprop (`exp_1_default_parity_*`)
`SimpleMoleculeMolGraphFeaturizer` is used with `ignore_stereo=False` (the default).
RDKit encodes the `@/@@` SMILES token as a `ChiralTag` attribute on the tetrahedral
carbon atom:

| SMILES token | RDKit ChiralTag | One-hot index |
|---|---|---|
| (unspecified) | `CHI_UNSPECIFIED` | 0 |
| `@@` | `CHI_TETRAHEDRAL_CW` | 1 |
| `@` | `CHI_TETRAHEDRAL_CCW` | 2 |

This one-hot is included in the atom feature vector fed to the MPNN. The message passing layers propagate atom-level `@/@@` information across multiple bond hops. However, the permutation-invariant mean or sum aggregation over neighbors may dilute this signal.
The key question is whether this atomic-level stereo signal survives Chemprop's **permutation-invariant
mean aggregation** to produce a useful graph-level representation.

In [7]:
_feat_inspect = featurizers.SimpleMoleculeMolGraphFeaturizer()
from rdkit import Chem

_smi = df['SMILES'].iloc[0]
_mol = Chem.MolFromSmiles(_smi)
_graph = _feat_inspect(_mol)

print(f'Atom feature dim: {_graph.V.shape[1]}')
print(f'Bond feature dim: {_graph.E.shape[1]}')
print(f'Nodes:            {_graph.V.shape[0]}')

Atom feature dim: 72
Bond feature dim: 14
Nodes:            17


In [8]:
# Exp 1: standard stereo-aware datapoints (ignore_stereo=False)
all_data_stereo = [
    data.MoleculeDatapoint.from_smi(smi, ignore_stereo=False)
    for smi in df['SMILES']
]

print(f'Built {len(all_data_stereo):,} stereo datapoints')

Built 3,858 stereo datapoints


## Model and trainer builder functions

### Message Passing
`nn.BondMessagePassing()` — standard Chemprop bond-based message passing.

### Aggregation
`nn.MeanAggregation()` — permutation-invariant mean over node representations.
This is the architectural bottleneck for chirality: two enantiomers with different
`ChiralTag` atom features can still produce similar aggregated graph embeddings
if the mean washes out the sign information.

### Feed-Forward Network
`nn.BinaryClassificationFFN(n_tasks=1)` for single-task; `n_tasks=3` for multi-task.

In [9]:
def build_mpnn(n_tasks: int = 1) -> models.MPNN:
    """
    Build a fresh Chemprop v2 MPNN for binary classification.

    A new model is instantiated for every (condition, target, fold) to
    ensure no weight sharing across runs.

    For the BinaryClassificationFFN, n_targets is hardcoded to 1 and the default loss is BCELoss
    https://github.com/chemprop/chemprop/blob/7033cac6ebc1a66f885705c7fa3e2cb960f96e49/chemprop/nn/predictors.py#L236

    Passing n_tasks=3 means the output dimension becomes a tensor of shape (batch_size, 3).
    A .sigmoid() activation is applied to the entire tensor so each of the 3 targets is evaluated
    independently and gets its own distinct probability between 0 and 1.

    Args:
        n_tasks: Number of binary classification outputs. 1 for single-task;
            3 for the multi-task condition predicting all targets jointly.

    Returns:
        Configured chemprop.models.MPNN ready for training.
    """
    mp  = nn.BondMessagePassing()
    agg = nn.MeanAggregation()
    ffn = nn.BinaryClassificationFFN(n_tasks=n_tasks)  
    metric_list = [
        nn.metrics.BinaryAUROC(),    # primary — used for EarlyStopping
        nn.metrics.BinaryAUPRC(),
        nn.metrics.BinaryAccuracy(),
        nn.metrics.BinaryF1Score(),
        # MCC is not natively available in Chemprop; computed post-hoc with sklearn
    ]
    return models.MPNN(mp, agg, ffn, batch_norm=False, metrics=metric_list)


def build_trainer(fold_idx: int, condition_name: str, target_col: str) -> pl.Trainer:
    """
    Build a Lightning Trainer with EarlyStopping and ModelCheckpoint.

    EarlyStopping monitors val/roc (BinaryAUROC) and stops if it does not improve
    for PATIENCE consecutive epochs.

    Args:
        fold_idx: The current fold number (used for checkpoint naming).
        condition_name: The featurization condition (used for checkpoint naming).
        target_col: The target being predicted (used for checkpoint naming).

    Returns:
        Configured pl.Trainer.
    """
    # must use `monitor='val/roc'`, as logged by Chemprop
    # monitor='val_BinaryAUROC' causes RuntimeError: Early stopping conditioned on metric
    # `val_BinaryAUROC` which is not available. Use any of: `train_loss`, `val/roc`, etc.
    early_stop = EarlyStopping(
        monitor='val/roc',
        patience=PATIENCE,
        mode='max',
        verbose=False,
    )

    # Add ModelCheckpoint to save the best model
    checkpoint_callback = ModelCheckpoint(
        dirpath='checkpoints/',
        filename=f"{condition_name}_{target_col.replace('/', '_')}_fold{fold_idx}_best",
        monitor='val/roc',
        mode='max',
        save_top_k=1, # Only save the best one
        verbose=False,
    )

    return pl.Trainer(
        logger=False,
        enable_checkpointing=True,
        enable_progress_bar=False,
        accelerator='auto',
        devices=1,
        max_epochs=MAX_EPOCHS,
        callbacks=[early_stop, checkpoint_callback],
    )

## `evaluate_chemprop` helper

Loads the best checkpoint and computes all metrics for a single loader/split.
MCC is computed post-hoc via sklearn (not natively available in Chemprop v2).

In [10]:
def evaluate_chemprop(
    trainer: pl.Trainer,
    mpnn: models.MPNN,
    loader,
    y_true: np.ndarray,
) -> dict:
    """
    Run prediction and compute all metrics for one fold/split.

    MCC is not natively available in Chemprop v2 metrics, so it is computed
    post-hoc using sklearn at threshold 0.5, consistent with the RF notebooks.

    Args:
        trainer: Fitted pl.Trainer.
        mpnn: Fitted MPNN model.
        loader: DataLoader to run predictions on.
        y_true: 1D numpy array of true binary labels.

    Returns:
        Dict mapping metric name to scalar value.
    """
    # ckpt_path='best' loads the best checkpoint saved by ModelCheckpoint.
    # weights_only=False is required to load Chemprop's custom metric objects (BinaryAUROC etc.) from the PyTorch Lightning checkpoint without an UnpicklingError.
    preds = trainer.predict(mpnn, loader, ckpt_path='best', weights_only=False)
    probs = torch.cat(preds).squeeze().numpy()
    preds_bin = (probs > 0.5).astype(int)
    return {
        'AUROC':    roc_auc_score(y_true, probs),
        'MCC':      matthews_corrcoef(y_true, preds_bin),
        'Accuracy': accuracy_score(y_true, preds_bin),
        'F1':       f1_score(y_true, preds_bin),
        'AUPRC':    average_precision_score(y_true, probs),
    }

## Exp 1 — Single-task training loop (default parity, implicit Hs)

1 condition × 3 targets × 5 folds = **15 total fits**

In [11]:
# ── Exp 1: default parity encoding ──────────────────────────────────────────
# 1 condition x 3 targets x 5 folds = 15 fits
SINGLE_TASK_CONDITIONS = {
    'exp_1_default_parity_singletask': all_data_stereo,
}

featurizer = featurizers.SimpleMoleculeMolGraphFeaturizer()
all_results = []

for condition_name, all_dpoints in SINGLE_TASK_CONDITIONS.items():
    CONDITION_START = time.time()
    print(f"\n{'#'*70}")
    print(f'Condition: {condition_name}')
    print(f"{'#'*70}")

    for target_col in SINGLE_TARGETS:
        label_col = f'label_{target_col}'
        print(f"\n  Target: {target_col}\n  {'-'*60}")

        for fold_idx, fold in enumerate(mol_folds):
            FOLD_START = time.time()
            tr_pts = [all_dpoints[i] for i in fold['train']]
            va_pts = [all_dpoints[i] for i in fold['val']]
            te_pts = [all_dpoints[i] for i in fold['test']]

            y_train = df[label_col].iloc[fold['train']].values.reshape(-1, 1)
            y_val   = df[label_col].iloc[fold['val']].values.reshape(-1, 1)
            y_test  = df[label_col].iloc[fold['test']].values.reshape(-1, 1)

            # Attach labels to datapoints for this run
            for dp, yi in zip(tr_pts, y_train): dp.y = yi
            for dp, yi in zip(va_pts, y_val):   dp.y = yi
            for dp, yi in zip(te_pts, y_test):  dp.y = yi

            train_dset = data.MoleculeDataset(tr_pts, featurizer)
            val_dset   = data.MoleculeDataset(va_pts, featurizer)
            test_dset  = data.MoleculeDataset(te_pts, featurizer)

            train_loader = data.build_dataloader(
                train_dset, batch_size=BATCH_SIZE,
                num_workers=NUM_WORKERS, shuffle=True)
            val_loader = data.build_dataloader(
                val_dset, batch_size=BATCH_SIZE,
                num_workers=NUM_WORKERS, shuffle=False)
            test_loader = data.build_dataloader(
                test_dset, batch_size=BATCH_SIZE,
                num_workers=NUM_WORKERS, shuffle=False)

            mpnn = build_mpnn(n_tasks=1)
            trainer = build_trainer(fold_idx, condition_name, target_col)
            trainer.fit(mpnn, train_loader, val_loader)

            splits_to_eval = [
                ('train', train_loader, y_train.ravel()),
                ('val',   val_loader,   y_val.ravel()),
                ('test',  test_loader,  y_test.ravel()),
            ]

            for split_name, loader, y_true in splits_to_eval:
                metrics = evaluate_chemprop(trainer, mpnn, loader, y_true)
                metrics.update({
                    'fold':          fold_idx,
                    'model':         'chemprop',
                    'featurization': condition_name,
                    'target':        target_col,
                    'split':         split_name,
                    'stopped_epoch': trainer.current_epoch,
                })
                all_results.append(metrics)

                if split_name == 'test':
                    fold_elapsed = time.time() - FOLD_START
                    print(
                        f'  fold {fold_idx} (test) | '
                        f"AUROC={metrics['AUROC']:.3f}  "
                        f"MCC={metrics['MCC']:.3f}  "
                        f"Acc={metrics['Accuracy']:.3f}  "
                        f"stopped_epoch={metrics['stopped_epoch']}  "
                        f"fold_time={fold_elapsed:.1f}s"
                    )

            # free up memory
            # delete the Python references to the large objects
            del mpnn, trainer
            # force Python's Garbage Collector to immediately clean up RAM
            gc.collect()
            # dynamically clear the hardware accelerator cache
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            elif torch.backends.mps.is_available():
                torch.mps.empty_cache()

    total_elapsed = time.time() - CONDITION_START
    print(f'\nTotal time for {condition_name}: {total_elapsed:.1f}s ({total_elapsed/60:.1f} min)')

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
Loading `train_dataloader` to estimate number of stepping batches.



######################################################################
Condition: exp_1_default_parity_singletask
######################################################################

  Target: R/S_class
  ------------------------------------------------------------


/Users/kevin/opt/anaconda3/envs/chemprop_v2/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type                    ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ message_passing │ BondMessagePassing      │  227 K │ train │     0 │
│ 1 │ agg             │ MeanAggregation         │      0 │ train │     0 │
│ 2 │ bn              │ Identity                │      0 │ train │     0 │
│ 3 │ predictor       │ BinaryClassificationFFN │ 90.6 K │ train │     0 │
│ 4 │ X_d_transform   │ Identity                │      0 │ train │     0 │
│ 5 │ metrics         │ ModuleList              │      0 │ train │     0 │
└───┴─────────────────┴─────────────────────────┴────────┴───────┴───────┘

Trainable params: 318 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 318 K                                                                                                
Total estimated model params size (MB): 1                                                                          
Modules in train mode: 27                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

...

/Users/kevin/opt/anaconda3/envs/chemprop_v2/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type                    ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ message_passing │ BondMessagePassing      │  227 K │ train │     0 │
│ 1 │ agg             │ MeanAggregation         │      0 │ train │     0 │
│ 2 │ bn              │ Identity                │      0 │ train │     0 │
│ 3 │ predictor       │ BinaryClassificationFFN │ 90.6 K │ train │     0 │
│ 4 │ X_d_transform   │ Identity                │      0 │ train │     0 │
│ 5 │ metrics         │ ModuleList              │      0 │ train │     0 │
└───┴─────────────────┴─────────────────────────┴────────┴───────┴───────┘

Trainable params: 318 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 318 K                                                                                                
Total estimated model params size (MB): 1                                                                          
Modules in train mode: 27                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Restoring states from the checkpoint path at /Users/kevin/Documents/code/stereoisomer_prediction/cmrt/checkpoints/exp_1_default_parity_singletask_F_L_class_fold4_best.ckpt
Loaded model weights from the checkpoint at /Users/kevin/Documents/code/stereoisomer_prediction/cmrt/checkpoints/exp_1_default_parity_singletask_F_L_class_fold4_best.ckpt
Restoring states from the checkpoint path at /Users/kevin/Documents/code/stereoisomer_prediction/cmrt/checkpoints/exp_1_default_parity_singletask_F_L_class_fold4_best.ckpt
Loaded model weights from the checkpoint at /Users/kevin/Documents/code/stereoisomer_prediction/cmrt/checkpoints/exp_1_default_parity_singletask_F_L_class_fold4_best.ckpt
/Users/kevin/opt/anaconda3/envs/chemprop_v2/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
Restoring states from the checkpoint path at /Users/kevin/Documents/code/stereois

  fold 4 (test) | AUROC=0.798  MCC=0.455  Acc=0.726  stopped_epoch=36  fold_time=172.7s

Total time for exp_1_default_parity_singletask: 4058.8s (67.6 min)


## Exp 1 — Multi-task loop (default parity, implicit Hs)

Joint prediction of all 3 targets. **5 total fits.**

Tests whether shared MPNN representations across targets improve per-target performance.

In [12]:
# ── Exp 1 multitask ──────────────────────────────────────────────────────────
# Default parity featurizer, joint prediction of all 3 targets.
# 5 folds = 5 fits

print(f"\n{'#'*70}")
MULTITASK_CONDITION = 'exp_1_default_parity_multitask'
print(f'Condition: {MULTITASK_CONDITION}')
print(f"{'#'*70}")

CONDITION_START = time.time()

for fold_idx, fold in enumerate(mol_folds):
    FOLD_START = time.time()
    print(f"\n  Fold {fold_idx}  |  "
          f"train={len(fold['train']):,}  "
          f"val={len(fold['val']):,}  "
          f"test={len(fold['test']):,}")

    tr_pts = [all_data_stereo[i] for i in fold['train']]
    va_pts = [all_data_stereo[i] for i in fold['val']]
    te_pts = [all_data_stereo[i] for i in fold['test']]

    label_cols = [f'label_{t}' for t in SINGLE_TARGETS]
    y_train = df[label_cols].iloc[fold['train']].values
    y_val   = df[label_cols].iloc[fold['val']].values
    y_test  = df[label_cols].iloc[fold['test']].values

    for dp, yi in zip(tr_pts, y_train): dp.y = yi
    for dp, yi in zip(va_pts, y_val):   dp.y = yi
    for dp, yi in zip(te_pts, y_test):  dp.y = yi

    train_dset = data.MoleculeDataset(tr_pts, featurizer)
    val_dset   = data.MoleculeDataset(va_pts, featurizer)
    test_dset  = data.MoleculeDataset(te_pts, featurizer)

    train_loader = data.build_dataloader(
        train_dset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, shuffle=True)
    val_loader   = data.build_dataloader(
        val_dset,   batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, shuffle=False)
    test_loader  = data.build_dataloader(
        test_dset,  batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, shuffle=False)

    mpnn    = build_mpnn(n_tasks=3)
    trainer = build_trainer(fold_idx, MULTITASK_CONDITION, 'all_targets')
    trainer.fit(mpnn, train_loader, val_loader)

    splits_to_eval = [
        ('train', train_loader, y_train),
        ('val',   val_loader,   y_val),
        ('test',  test_loader,  y_test),
    ]

    for split_name, loader, y_true_all in splits_to_eval:
        preds     = trainer.predict(mpnn, loader, ckpt_path='best', weights_only=False)
        all_probs = torch.cat(preds).numpy()

        for task_idx, target_col in enumerate(SINGLE_TARGETS):
            probs     = all_probs[:, task_idx]
            y_true    = y_true_all[:, task_idx]
            preds_bin = (probs > 0.5).astype(int)
            metrics = {
                'AUROC':    roc_auc_score(y_true, probs),
                'MCC':      matthews_corrcoef(y_true, preds_bin),
                'Accuracy': accuracy_score(y_true, preds_bin),
                'F1':       f1_score(y_true, preds_bin),
                'AUPRC':    average_precision_score(y_true, probs),
                'fold':          fold_idx,
                'model':         'chemprop',
                'featurization': MULTITASK_CONDITION,
                'target':        target_col,
                'split':         split_name,
                'stopped_epoch': trainer.current_epoch,
            }
            all_results.append(metrics)

            if split_name == 'test':
                print(
                    f"  {target_col} (test): "
                    f"AUROC={metrics['AUROC']:.3f}  "
                    f"MCC={metrics['MCC']:.3f}  "
                    f"Acc={metrics['Accuracy']:.3f}"
                )

    fold_elapsed = time.time() - FOLD_START
    print(f'  fold_time={fold_elapsed:.1f}s')

    del mpnn, trainer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    elif torch.backends.mps.is_available():
        torch.mps.empty_cache()

total_elapsed = time.time() - CONDITION_START
print(f'\nTotal time for {MULTITASK_CONDITION}: {total_elapsed:.1f}s ({total_elapsed/60:.1f} min)')


GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
Loading `train_dataloader` to estimate number of stepping batches.



######################################################################
Condition: exp_1_default_parity_multitask
######################################################################

  Fold 0  |  train=3,478  val=190  test=190


/Users/kevin/opt/anaconda3/envs/chemprop_v2/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type                    ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ message_passing │ BondMessagePassing      │  227 K │ train │     0 │
│ 1 │ agg             │ MeanAggregation         │      0 │ train │     0 │
│ 2 │ bn              │ Identity                │      0 │ train │     0 │
│ 3 │ predictor       │ BinaryClassificationFFN │ 91.2 K │ train │     0 │
│ 4 │ X_d_transform   │ Identity                │      0 │ train │     0 │
│ 5 │ metrics         │ ModuleList              │      0 │ train │     0 │
└───┴─────────────────┴─────────────────────────┴────────┴───────┴───────┘

Trainable params: 318 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 318 K                                                                                                
Total estimated model params size (MB): 1                                                                          
Modules in train mode: 27                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

...

/Users/kevin/opt/anaconda3/envs/chemprop_v2/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type                    ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ message_passing │ BondMessagePassing      │  227 K │ train │     0 │
│ 1 │ agg             │ MeanAggregation         │      0 │ train │     0 │
│ 2 │ bn              │ Identity                │      0 │ train │     0 │
│ 3 │ predictor       │ BinaryClassificationFFN │ 91.2 K │ train │     0 │
│ 4 │ X_d_transform   │ Identity                │      0 │ train │     0 │
│ 5 │ metrics         │ ModuleList              │      0 │ train │     0 │
└───┴─────────────────┴─────────────────────────┴────────┴───────┴───────┘

Trainable params: 318 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 318 K                                                                                                
Total estimated model params size (MB): 1                                                                          
Modules in train mode: 27                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

`Trainer.fit` stopped: `max_epochs=50` reached.
Restoring states from the checkpoint path at /Users/kevin/Documents/code/stereoisomer_prediction/cmrt/checkpoints/exp_1_default_parity_multitask_all_targets_fold4_best.ckpt
Loaded model weights from the checkpoint at /Users/kevin/Documents/code/stereoisomer_prediction/cmrt/checkpoints/exp_1_default_parity_multitask_all_targets_fold4_best.ckpt
Restoring states from the checkpoint path at /Users/kevin/Documents/code/stereoisomer_prediction/cmrt/checkpoints/exp_1_default_parity_multitask_all_targets_fold4_best.ckpt
Loaded model weights from the checkpoint at /Users/kevin/Documents/code/stereoisomer_prediction/cmrt/checkpoints/exp_1_default_parity_multitask_all_targets_fold4_best.ckpt
/Users/kevin/opt/anaconda3/envs/chemprop_v2/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
Restoring states from the che

  R/S_class (test): AUROC=0.799  MCC=0.516  Acc=0.758
  @/@@_class (test): AUROC=0.989  MCC=0.916  Acc=0.958
  F/L_class (test): AUROC=0.786  MCC=0.442  Acc=0.721
  fold_time=212.5s

Total time for exp_1_default_parity_multitask: 1067.9s (17.8 min)


## Summary across Exp 1 conditions

In [13]:
results_df = pd.DataFrame(all_results)
metric_cols  = ['AUROC', 'MCC', 'Accuracy', 'F1', 'AUPRC']
target_order = ['R/S_class', '@/@@_class', 'F/L_class']
condition_order = [
    'exp_1_default_parity_singletask',
    'exp_1_default_parity_multitask',
]

test_results = results_df[results_df['split'] == 'test']
summary = (
    test_results
    .groupby(['featurization', 'target'])[metric_cols]
    .agg(['mean', 'std'])
    .round(4)
)
summary = summary.reindex([(c, t) for c in condition_order for t in target_order])
print('Test Set Performance (Mean ± std across 5 folds):')
print(summary.to_string())

Test Set Performance (Mean ± std across 5 folds):
                                             AUROC             MCC         Accuracy              F1           AUPRC        
                                              mean     std    mean     std     mean     std    mean     std    mean     std
featurization                   target                                                                                     
exp_1_default_parity_singletask R/S_class   0.8233  0.0607  0.4760  0.1140   0.7379  0.0570  0.7375  0.0568  0.8205  0.0715
                                @/@@_class  0.9926  0.0050  0.9167  0.0519   0.9579  0.0268  0.9587  0.0252  0.9928  0.0048
                                F/L_class   0.7628  0.0466  0.3756  0.1148   0.6874  0.0572  0.6818  0.0612  0.7698  0.0435
exp_1_default_parity_multitask  R/S_class   0.8162  0.0622  0.4950  0.1033   0.7474  0.0517  0.7454  0.0573  0.8181  0.0594
                                @/@@_class  0.9905  0.0062  0.9116  0.0338   0.955

## Hypothesis evaluation

Two comparisons:
1. **@/@@ vs R/S** for singletask — tests the ChiralTag encoding hypothesis.
2. **Multitask vs singletask** — tests shared representation benefit.

In [14]:
# ── Hypothesis 1: @/@@ should be easier than R/S for Chemprop ────────────────
# Chemprop encodes @/@@ via ChiralTag; RF encoded R/S. If the signal survives
# aggregation, @/@@ should be the easiest target here (inverse of RF result).
df_st_test = results_df[
    (results_df['featurization'] == 'exp_1_default_parity_singletask') &
    (results_df['split'] == 'test')
]

at_auroc = df_st_test[df_st_test['target'] == '@/@@_class']['AUROC'].mean()
rs_auroc = df_st_test[df_st_test['target'] == 'R/S_class']['AUROC'].mean()

print('Hypothesis 1: @/@@ easier than R/S for Chemprop (encodes @/@@ via ChiralTag)?')
print(f'  @/@@ AUROC (singletask): {at_auroc:.4f}')
print(f'  R/S  AUROC (singletask): {rs_auroc:.4f}')
if at_auroc > rs_auroc:
    print('  CONFIRMED: @/@@ > R/S — consistent with ChiralTag encoding @/@@')
else:
    print('  NOT CONFIRMED: R/S >= @/@@ — MPNN may lose @/@@ during mean aggregation')

print()

# ── Multitask vs singletask comparison ────────────────────────────────────────
df_mt_test = results_df[
    (results_df['featurization'] == 'exp_1_default_parity_multitask') &
    (results_df['split'] == 'test')
]

print('Multitask vs singletask AUROC (test set):')
print(f"  {'Target':<15} {'Singletask':>12} {'Multitask':>12} {'Delta':>8}")
for target in target_order:
    st = df_st_test[df_st_test['target'] == target]['AUROC'].mean()
    mt = df_mt_test[df_mt_test['target'] == target]['AUROC'].mean()
    print(f"  {target:<15} {st:>12.4f} {mt:>12.4f} {mt - st:>+8.4f}")

Hypothesis 1: @/@@ easier than R/S for Chemprop (encodes @/@@ via ChiralTag)?
  @/@@ AUROC (singletask): 0.9926
  R/S  AUROC (singletask): 0.8233
  CONFIRMED: @/@@ > R/S — consistent with ChiralTag encoding @/@@

Multitask vs singletask AUROC (test set):
  Target            Singletask    Multitask    Delta
  R/S_class             0.8233       0.8162  -0.0072
  @/@@_class            0.9926       0.9905  -0.0020
  F/L_class             0.7628       0.7454  -0.0174


## Per-condition performance table (test set)

In [15]:
print(f"{'Target':<15} {'Condition':<40} {'AUROC':>7} {'MCC':>7} {'Acc':>7}")
print('-' * 80)
for target in target_order:
    for cond in condition_order:
        s = results_df[
            (results_df['target'] == target) &
            (results_df['featurization'] == cond) &
            (results_df['split'] == 'test')
        ]
        if s.empty:
            continue
        auroc = s['AUROC'].mean()
        mcc   = s['MCC'].mean()
        acc   = s['Accuracy'].mean()
        print(f'{target:<15} {cond:<40} {auroc:>7.4f} {mcc:>7.4f} {acc:>7.4f}')
    print()

Target          Condition                                  AUROC     MCC     Acc
--------------------------------------------------------------------------------
R/S_class       exp_1_default_parity_singletask           0.8233  0.4760  0.7379
R/S_class       exp_1_default_parity_multitask            0.8162  0.4950  0.7474

@/@@_class      exp_1_default_parity_singletask           0.9926  0.9167  0.9579
@/@@_class      exp_1_default_parity_multitask            0.9905  0.9116  0.9558

F/L_class       exp_1_default_parity_singletask           0.7628  0.3756  0.6874
F/L_class       exp_1_default_parity_multitask            0.7454  0.3389  0.6684



## Early stopping epoch distribution

In [16]:
epoch_summary = (
    results_df[results_df['split'] == 'test']
    .groupby(['featurization', 'target'])['stopped_epoch']
    .agg(['mean', 'min', 'max'])
    .round(1)
)
print('Epochs trained before early stopping:')
print(epoch_summary.to_string())

Epochs trained before early stopping:
                                            mean  min  max
featurization                   target                    
exp_1_default_parity_multitask  @/@@_class  50.0   50   50
                                F/L_class   50.0   50   50
                                R/S_class   50.0   50   50
exp_1_default_parity_singletask @/@@_class  38.6   25   50
                                F/L_class   45.4   36   50
                                R/S_class   38.0   24   50


## Save results for Tukey HSD

Per-fold scores are saved to CSV for downstream statistical comparison.
Load alongside `4_chemprop_exp_0_results.csv` and RF results for cross-model Tukey HSD.

In [17]:
output_path = '4_chemprop_exp_1_results.csv'
results_df.to_csv(output_path, index=False)
print(f'\nSaved {len(results_df)} total rows to {output_path}')
print(f"  Conditions: {results_df['featurization'].unique().tolist()}")
print(f"  Targets:    {results_df['target'].unique().tolist()}")
print(f"  Folds:      {sorted(results_df['fold'].unique().tolist())}")
print(f"  Splits:     {results_df['split'].unique().tolist()}")
print(f"  Columns:    {results_df.columns.tolist()}")


Saved 90 total rows to 4_chemprop_exp_1_results.csv
  Conditions: ['exp_1_default_parity_singletask', 'exp_1_default_parity_multitask']
  Targets:    ['R/S_class', '@/@@_class', 'F/L_class']
  Folds:      [0, 1, 2, 3, 4]
  Splits:     ['train', 'val', 'test']
  Columns:    ['AUROC', 'MCC', 'Accuracy', 'F1', 'AUPRC', 'fold', 'model', 'featurization', 'target', 'split', 'stopped_epoch']
